In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv('common_passwords.csv')

# -----------------------------
# Create Balanced Classification Target
# -----------------------------
# Relaxed rule: password is "strong" if it satisfies at least 2 conditions
df['is_strong'] = (
    (df['length'] >= 8).astype(int) +
    (df['num_digits'] > 0).astype(int) +
    (df['num_special'] > 0).astype(int) +
    (df['num_upper'] > 0).astype(int)

) >= 2

df['is_strong'] = df['is_strong'].astype(int)

# Check class distribution
print("Class distribution:")
print(df['is_strong'].value_counts())

# -----------------------------
# Features and Target
# -----------------------------
X = df.drop(columns=['password', 'is_strong'])
y = df['is_strong']

# -----------------------------
# Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# Feature Scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# Models
# -----------------------------
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(probability=True)

models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "SVM": svm
}

# -----------------------------
# Evaluation Function
# -----------------------------
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))
    print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# -----------------------------
# Train & Evaluate Models
# -----------------------------
for name, model in models.items():
    evaluate_model(name, model)

# -----------------------------
# Cross Validation
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-validation results:")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f"{name}: Mean Accuracy = {scores.mean():.4f}")

# -----------------------------
# Ensemble Method (Voting Classifier)
# -----------------------------
ensemble = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('rf', rf),
        ('svm', svm)
    ],
    voting='soft'
)

evaluate_model("Ensemble (Voting Classifier)", ensemble)


Class distribution:
is_strong
0    8217
1    1783
Name: count, dtype: int64

Logistic Regression
----------------------------------------
Accuracy: 0.963
Precision: 0.9077809798270894
Recall: 0.8823529411764706
F1 Score: 0.8948863636363636
Confusion Matrix:
 [[1611   32]
 [  42  315]]

Random Forest
----------------------------------------
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[1643    0]
 [   0  357]]

SVM
----------------------------------------
Accuracy: 0.998
Precision: 1.0
Recall: 0.988795518207283
F1 Score: 0.9943661971830986
Confusion Matrix:
 [[1643    0]
 [   4  353]]

Cross-validation results:
Logistic Regression: Mean Accuracy = 0.9622
Random Forest: Mean Accuracy = 0.9997
SVM: Mean Accuracy = 0.9971

Ensemble (Voting Classifier)
----------------------------------------
Accuracy: 0.999
Precision: 1.0
Recall: 0.9943977591036415
F1 Score: 0.9971910112359551
Confusion Matrix:
 [[1643    0]
 [   2  355]]
